# SPOT-RNA
<!-- SPDX-License-Identifier: GPL-3.0-only -->

Adapted from `sinc-lab/lncRNA-folding`.
Modified by Jingwen Liu, 2026, for full-length viral RNA benchmarking.

In [ ]:
import os
import pandas as pd
import time
import shutil
import subprocess
from pathlib import Path
import sys
from contextlib import redirect_stdout, redirect_stderr
import io

In [ ]:
method_name = "SPOT-RNA"
base = Path.cwd()

spot_rna_base = base.parent / 'tools' / 'spot-rna'
spot_rna_env = spot_rna_base / 'env'

In [ ]:
# Clone SPOT-RNA if needed
if not (spot_rna_base / 'SPOT-RNA').exists():
    print("Cloning SPOT-RNA source code...")
    os.makedirs(str(spot_rna_base), exist_ok=True)
    os.chdir(str(spot_rna_base))
    !git clone --quiet https://github.com/jaswindersingh2/SPOT-RNA.git
    os.chdir(str(base))
    print("SPOT-RNA cloned successfully")
else:
    print(f"SPOT-RNA already exists at {spot_rna_base}")

In [ ]:
# Create conda environment if needed
if not spot_rna_env.exists():
    print("Creating SPOT-RNA conda environment...")
    !conda create --prefix {spot_rna_env} python=3.6 -y > /dev/null
    print("SPOT-RNA environment created successfully")
else:
    print(f"SPOT-RNA environment already exists")


In [ ]:
# Download models and install packages
spot_rna_src = spot_rna_base / 'SPOT-RNA'
if not (spot_rna_src / 'SPOT-RNA-models').exists():
    print("Downloading SPOT-RNA models...")
    os.chdir(str(spot_rna_src))
    !wget  'https://www.dropbox.com/s/dsrcf460nbjqpxa/SPOT-RNA-models.tar.gz'
    !wget  -O SPOT-RNA-models.tar.gz 'https://app.nihaocloud.com/f/fbf3315a91d542c0bdc2/?dl=1'
    !tar -xzf SPOT-RNA-models.tar.gz && rm SPOT-RNA-models.tar.gz
    os.chdir(str(base))
    print("SPOT-RNA models installed successfully")


In [ ]:
# Install dependencies (only if not already installed)
import subprocess
result = subprocess.run(f"conda run -p {spot_rna_env} pip show tensorflow-gpu", shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print("Installing SPOT-RNA dependencies...")
    !conda run -p {spot_rna_env} pip install -q tensorflow-gpu==1.14.0
    !conda run -p {spot_rna_env} pip install -q -r {spot_rna_src}/requirements.txt
    print("SPOT-RNA dependencies installed successfully")
else:
    print("SPOT-RNA dependencies already installed")

In [ ]:
def read_virus_fasta(path: str):
    lines = [ln.strip() for ln in open(path, 'r').read().splitlines() if ln.strip() != '']
    records = []
    for i in range(0, len(lines), 3):
        header, seq, struct = lines[i], lines[i+1], lines[i+2]
        name = header[1:].strip()
        records.append((name, seq.strip(), struct.strip()))
    df = pd.DataFrame(records, columns=['name','sequence','structure']).set_index('name')
    return df

viruses = read_virus_fasta('../data/viruses.fasta')

selected_virus_keys = None

if selected_virus_keys is None:
    virus_ids = list(viruses.index)
else:
    tmp = []
    for k in selected_virus_keys:
        if isinstance(k, int):
            tmp.append(viruses.index[k])
        else:
            tmp.append(str(k))
    virus_ids = tmp

In [ ]:
def run_folding(fasta_name):
  input_path = base / fasta_name
  original_dir = os.getcwd()
  os.chdir(str(spot_rna_src))

  out_file_name = base / "tmp_out.fasta"
  if os.path.exists("outputs"):
    shutil.rmtree("outputs")
  os.mkdir("outputs")

  with open(str(input_path)) as fin:
    seq_id = fin.readline().strip("> \n")
    sequence = fin.readline().strip()

  cmd = f"conda run -p {spot_rna_env} python SPOT-RNA.py --inputs {input_path} --outputs outputs --gpu 0"
  result = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
  
  # Debug: print stdout and stderr if command failed
  if result.returncode != 0:
    print(f"[DEBUG] SPOT-RNA.py failed with code {result.returncode}")
    print(f"[DEBUG] stderr: {result.stderr[:200]}")
    print(f"[DEBUG] stdout: {result.stdout[:200]}")

  # Find the actual .ct file in outputs directory
  output_files = os.listdir("outputs")
  ct_files = [f for f in output_files if f.endswith(".ct")]
  if not ct_files:
    print(f"ERR: No .ct: {output_files}")
    os.chdir(original_dir)
    return None

  ct_file = ct_files[0]
  dot_file = f"tmp_{seq_id}.dot"

  # Convert CT to dot-bracket notation using ct2dot.py
  convert_cmd = f"python {base}/ct2dot.py outputs/{ct_file} {dot_file} -f full -q"
  convert_result = subprocess.run(convert_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

  if convert_result.returncode != 0:
    print(f"ct2dot.py failed: {convert_result.stderr}")
    os.chdir(original_dir)
    return None

  # Read structure from dot file (format: line1=title, line2=sequence, line3=structure)
  structure = None
  if os.path.exists(dot_file):
    with open(dot_file) as f:
      lines = f.readlines()
      if len(lines) >= 3:
        structure = lines[2].strip()
  
  # Clean up temporary dot file
  if os.path.exists(dot_file):
    os.remove(dot_file)

  os.chdir(original_dir)

  if structure is None:
    print(f"ERROR: Failed to extract structure from {ct_file}")
    return None

  with open(str(out_file_name), "w") as fout:
    fout.write(f">{seq_id}\n")
    fout.write(f"{sequence}\n")
    fout.write(f"{structure}\n")

  return str(out_file_name)

In [ ]:
output_dir = base.parent / 'prediction'
output_dir.mkdir(exist_ok=True)

out_fasta_name = output_dir / (method_name + ".fasta")

# Remove existing file if it exists
if os.path.exists(out_fasta_name):
    os.remove(out_fasta_name)

print(f"{' ':3}\t{'virus':<20}\t{'len':<5}\t{'time'}")
for i, vid in enumerate(virus_ids):
    start_time = time.time()
    seq = viruses.loc[vid]['sequence']
    print(f"{i+1:3d}/{len(virus_ids)}\t{vid:<20}\t{len(seq):<5}\t", end='', flush=True)

    # Write a one-sequence fasta
    with open("spotrna_tmp.fasta", "w") as ofile:
        ofile.write(f">{vid}\n{seq}\n")

    dot_file_name = run_folding("spotrna_tmp.fasta")

    # Concatenate outputs only if folding was successful
    if dot_file_name and os.path.exists(dot_file_name):
        os.system(f"cat {dot_file_name} >> {out_fasta_name}")
        os.remove(dot_file_name)
        elapsed_time = time.time() - start_time
        print(f"{elapsed_time: .1f} s")
    else:
        elapsed_time = time.time() - start_time
        print(f"Failed - {elapsed_time: .1f} s")

# Clean up temporary files
for temp_file in ["spotrna_tmp.fasta", "tmp_out.fasta"]:
    if os.path.exists(temp_file):
        os.remove(temp_file)

print(f"\nProcessing complete. Results saved to {out_fasta_name}")